In [1]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import EuroSAT
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms

In [2]:
class ResNet50_M3(nn.Module):
    def __init__(self):
        super().__init__()

        # Load the heavier ResNet-50 model
        self.model = resnet50(weights=ResNet50_Weights.DEFAULT)

        # Fine-tune everything initially
        for param in self.model.parameters():
            param.requires_grad = True

        # Freeze BatchNorm2d parameters to keep pre-trained image statistics stable
        for module in self.model.modules():
            if isinstance(module, nn.BatchNorm2d):
                for param in module.parameters():
                    param.requires_grad = False

        # Replace the final classification head for our 10 EuroSAT classes
        self.model.fc = nn.Sequential(
            nn.Linear(self.model.fc.in_features, 200),
            nn.ReLU(),
            nn.Dropout(p=0.3), # Increased dropout slightly to prevent overfitting

            nn.Linear(200, 100),
            nn.ReLU(),
            nn.Dropout(p=0.3),

            nn.Linear(100, 10)
        )

    def train(self, mode=True):
        super().train(mode)
        # Keep BatchNorm layers in evaluation mode during training
        for module in self.model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()
        return self

    def forward(self, x):
        return self.model(x)

In [3]:
BATCH_SIZE = 64
EPOCHS = 25 # Slightly higher max epochs because the scheduler will help us train longer safely
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs("/content/reports", exist_ok=True)
os.makedirs("/content/checkpoints", exist_ok=True)
os.makedirs("/content/data/processed", exist_ok=True)

In [4]:
weights = ResNet50_Weights.DEFAULT

# Advanced Augmentation Pipeline
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=45), # New: Rotate images up to 45 degrees
    weights.transforms()
])

dataset = EuroSAT(
    root="/content/data/raw",
    download=True,
    transform=transform
)

100.0%


In [5]:
with open("/content/data/processed/splits.json") as f:
    splits = json.load(f)

train_dataset = Subset(dataset, splits["train"])
val_dataset   = Subset(dataset, splits["val"])
test_dataset  = Subset(dataset, splits["test"])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

FileNotFoundError: [Errno 2] No such file or directory: '/content/data/processed/splits.json'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
model = ResNet50_M3().to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

# NEW: The Learning Rate Scheduler
# If validation loss doesn't improve for 2 epochs, cut the learning rate in half
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

In [ ]:
PATIENCE = 6 # Increased patience since the scheduler needs time to work
best_val_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_accuracy = correct / total

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            predictions = outputs.argmax(dim=1)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_accuracy = val_correct / val_total

    # Step the scheduler based on validation loss
    scheduler.step(val_loss)

    # Save logic
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
    }

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(checkpoint, "/content/checkpoints/resnet50_m3_best.pth")
        print(f"Epoch [{epoch + 1}/{EPOCHS}] Train Acc: {train_accuracy * 100:.2f}% | Val Acc: {val_accuracy * 100:.2f}% ← best")
    else:
        epochs_without_improvement += 1
        print(f"Epoch [{epoch + 1}/{EPOCHS}] Train Acc: {train_accuracy * 100:.2f}% | Val Acc: {val_accuracy * 100:.2f}% ({epochs_without_improvement}/{PATIENCE})")
        if epochs_without_improvement >= PATIENCE:
            print(f"\nEarly stopping triggered after {epoch + 1} epochs.")
            break

# Restore best model
best_checkpoint = torch.load("/content/checkpoints/resnet50_m3_best.pth", map_location=DEVICE)
model.load_state_dict(best_checkpoint["model_state_dict"])
print(f"\nBest ResNet-50 restored from epoch {best_checkpoint['epoch']} with validation accuracy {best_checkpoint['val_accuracy'] * 100:.2f}%")

In [ ]:
# TEST CELL
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        predictions = outputs.argmax(dim=1)
        y_true.extend(labels.numpy())
        y_pred.extend(predictions.cpu().numpy())

# REPORT CELL
report = classification_report(y_true, y_pred, target_names=dataset.classes)
cm = confusion_matrix(y_true, y_pred)

print("\nClassification Report (ResNet-50 Advanced):")
print(report)
print("\nConfusion Matrix:")
print(cm)

with open("/content/reports/resnet50_advanced_report.txt", "w") as f:
    f.write(report)
    f.write("\n\nConfusion Matrix:\n")
    f.write(str(cm))